# Multi-Modal Medical Diagnosis Agent — Full Pipeline
**University of Hertfordshire | Group Project**

This notebook covers **Phase 1 (Preprocessing) + EDA** for Chest XRay dataset:

| Dataset | Source | Used By |
|---------|--------|---------|
| 1 — NIH ChestX-ray14 | NIH Box | Model 1 — ChestVision |

# Install all required packages

In [1]:

!pip install -q datasets seaborn wordcloud scikit-learn joblib tqdm matplotlib pandas numpy
print('All packages installed')

All packages installed


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# Import Required Libraries

In [ ]:

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm


%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', font_scale=1.05)

PALETTE = {
    'blue':'#2196F3','green':'#4CAF50','orange':'#FF9800',
    'red':'#F44336','purple':'#9C27B0','teal':'#009688',
    'pink':'#E91E63','grey':'#607D8B',
}
BRAND = list(PALETTE.values())
RANDOM_SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
print('Imports done')

Imports done


In [3]:
# Path configuration
DRIVE_BASE = Path('')
RAW        = DRIVE_BASE / 'data' / 'raw'
PROC       = DRIVE_BASE / 'data' / 'processed'
FIG_DIR    = Path('figures')

for p in [RAW/'chestxray', RAW/'synthea', PROC/'chestxray',
          PROC/'synthea', PROC/'medqa', FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR/name, dpi=150, bbox_inches='tight')
    try: plt.savefig(DRIVE_BASE/'eda'/'figures'/name, dpi=150, bbox_inches='tight')
    except: pass
    plt.show()
    print(f'   💾 {name}')

print('Paths configured:')
for label, p in [('ChestXray raw', RAW/'chestxray'), ('Synthea raw', RAW/'synthea')]:
    status = '✅' if any(p.iterdir()) else '❌  EMPTY — upload files first'
    print(f'  {status}  {label}: {p}')
print('  ℹ️   MedQA: will be downloaded automatically from HuggingFace')

Paths configured:
  ✅  ChestXray raw: data\raw\chestxray
  ❌  EMPTY — upload files first  Synthea raw: data\raw\synthea
  ℹ️   MedQA: will be downloaded automatically from HuggingFace


# DATASET 1 — NIH ChestX-ray14
### Preprocessing
Parses label CSV, one-hot encodes 14 conditions, splits **by patient ID** to prevent leakage.

In [4]:
#  ChestXray: constants
CX_LABELS = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass',
    'Nodule','Pneumonia','Pneumothorax','Consolidation','Edema',
    'Emphysema','Fibrosis','Pleural_Thickening','Hernia','No Finding'
]
TARGET = ['Cardiomegaly','Effusion','Pneumonia','Consolidation','Edema',
          'Pneumothorax','Infiltration','Atelectasis','Nodule','Mass',
          'Pleural_Thickening','Emphysema','Fibrosis','Hernia','No Finding']

CX_CSV = RAW / 'chestxray' / 'Data_Entry_2017.csv'
CX_OUT = PROC / 'chestxray'

print('Target conditions:', TARGET)
print('CSV path:', CX_CSV)
print('Output  :', CX_OUT)

Target conditions: ['Cardiomegaly', 'Effusion', 'Pneumonia', 'Consolidation', 'Edema', 'Pneumothorax', 'Infiltration', 'Atelectasis', 'Nodule', 'Mass', 'Pleural_Thickening', 'Emphysema', 'Fibrosis', 'Hernia', 'No Finding']
CSV path: data\raw\chestxray\Data_Entry_2017.csv
Output  : data\processed\chestxray


# ChestXray: load & one-hot encode

In [5]:
def load_chestxray_labels(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    # One-hot encode all 15 classes (14 conditions + No Finding)
    for label in TARGET:
        if label == 'No Finding':
            df[label] = df['Finding Labels'].apply(
                lambda x: 1 if str(x).strip() == 'No Finding' else 0
            )
        else:
            df[label] = df['Finding Labels'].apply(
                lambda x: 1 if label in str(x).split('|') else 0
            )
    return df

def filter_chestxray(df):
    """Keep rows with at least one target class active (always true since TARGET covers all classes)."""
    mask = df[TARGET].sum(axis=1) > 0
    return df[mask].copy()

print('Loading ChestX-ray14 labels...')
cx_df = load_chestxray_labels(CX_CSV)
print(f'  Total records : {len(cx_df):,}')
cx_df = filter_chestxray(cx_df)
print(f'  After filter  : {len(cx_df):,}')

print('\nCondition prevalence:')
for c in TARGET:
    n = cx_df[c].sum()
    print(f'  {c:<22} {n:>6,}  ({n/len(cx_df)*100:.1f}%)')
cx_df.head(3)

Loading ChestX-ray14 labels...
  Total records : 112,120
  After filter  : 112,120

Condition prevalence:
  Cardiomegaly            2,776  (2.5%)
  Effusion               13,317  (11.9%)
  Pneumonia               1,431  (1.3%)
  Consolidation           4,667  (4.2%)
  Edema                   2,303  (2.1%)
  Pneumothorax            5,302  (4.7%)
  Infiltration           19,894  (17.7%)
  Atelectasis            11,559  (10.3%)
  Nodule                  6,331  (5.6%)
  Mass                    5,782  (5.2%)
  Pleural_Thickening      3,385  (3.0%)
  Emphysema               2,516  (2.2%)
  Fibrosis                1,686  (1.5%)
  Hernia                    227  (0.2%)
  No Finding             60,361  (53.8%)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Sex,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,...,Pneumothorax,Infiltration,Atelectasis,Nodule,Mass,Pleural_Thickening,Emphysema,Fibrosis,Hernia,No Finding
0,00000001_000.png,Cardiomegaly,0,1,57,M,PA,2682,2749,0.143,...,0,0,0,0,0,0,0,0,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,...,0,0,0,0,0,0,1,0,0,0
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,...,0,0,0,0,0,0,0,0,0,0


# ChestXray: patient-level split + save

In [6]:
def patient_split(df, seed=RANDOM_SEED):
    """Split by Patient ID — same patient's images never span train and test."""
    pids = df['Patient ID'].unique()
    train_ids, temp_ids = train_test_split(
        pids, test_size=VAL_RATIO+TEST_RATIO, random_state=seed
    )
    val_ids, test_ids = train_test_split(
        temp_ids, test_size=TEST_RATIO/(VAL_RATIO+TEST_RATIO), random_state=seed
    )
    id_map = ({p:'train' for p in train_ids}
              | {p:'val' for p in val_ids}
              | {p:'test' for p in test_ids})
    df = df.copy()
    df['split'] = df['Patient ID'].map(id_map)
    return df

print('Splitting by Patient ID (no data leakage)...')
cx_df = patient_split(cx_df)

for split in ['train','val','test']:
    subset = cx_df[cx_df['split']==split]
    subset.to_csv(CX_OUT/f'{split}.csv', index=False)
    print(f'  {split:6s}: {len(subset):,} images → saved')

# Verify no patient leaks across splits
train_pids = set(cx_df[cx_df['split']=='train']['Patient ID'])
test_pids  = set(cx_df[cx_df['split']=='test']['Patient ID'])
overlap    = train_pids & test_pids
print(f'\nPatient overlap train↔test: {len(overlap)} (should be 0) ', '✅' if len(overlap)==0 else '❌')
print('\n Dataset 1 preprocessing complete')

Splitting by Patient ID (no data leakage)...
  train : 78,566 images → saved
  val   : 17,063 images → saved
  test  : 16,491 images → saved

Patient overlap train↔test: 0 (should be 0)  ✅

 Dataset 1 preprocessing complete


---
## Key EDA Findings

| Dataset | Key Finding | Impact on Model |
|---------|------------|----------------|
| ChestXray | No_Finding = 70.7% — severe class imbalance | Use **Asymmetric Loss (ASL)** in ChestVision |
| ChestXray | Max 9 conditions per image; most have 1 | Multi-label sigmoid output, not softmax |